# Test Silver Transformation

Test notebook to preview Bronze → Silver transformation on a sample.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
import os
from dotenv import load_dotenv

load_dotenv()

spark = SparkSession.builder.getOrCreate()

STORAGE_ACCOUNT = os.getenv("DATALAKE_NAME")
ACCESS_KEY = os.getenv("DATALAKE_ACCESS_KEY")

BRONZE_PATH = f"abfss://bronze@{STORAGE_ACCOUNT}.dfs.core.windows.net/water_quality/hubeau"

spark.conf.set(f"fs.azure.account.key.{STORAGE_ACCOUNT}.dfs.core.windows.net", ACCESS_KEY)

print(f"Bronze: {BRONZE_PATH}")

In [ ]:
# Read Bronze sample directly from Parquet (limit 1000 records)
df_bronze = spark.read.parquet(BRONZE_PATH).limit(1000)

print(f"Sample size: {df_bronze.count()} records")
print("\nBRONZE Schema:")
df_bronze.printSchema()

In [ ]:
# Show sample BEFORE transformation
print("=== BEFORE (Bronze) ===")
df_bronze.select(
    "code_prelevement",
    "code_parametre",
    "libelle_parametre",
    "date_prelevement",
    "resultat_numerique",
    "libelle_unite",
    "code_departement",
    "nom_departement",
    "code_commune",
    "nom_commune",
    "conclusion_conformite_prelevement"
).show(5, truncate=False)

In [ ]:
# Apply Silver transformations

# 1. Fix data types
df_clean = df_bronze \
    .withColumn("resultat_numerique", F.col("resultat_numerique").cast(DoubleType())) \
    .withColumn("date_prelevement", F.to_timestamp("date_prelevement")) \
    .withColumn("code_departement", F.col("code_departement").cast(IntegerType())) \
    .withColumn("code_commune", F.col("code_commune").cast(IntegerType()))

print("✓ Data types fixed")

# 2. Remove duplicates
before_dedup = df_clean.count()
df_clean = df_clean.dropDuplicates([
    "code_prelevement",
    "code_parametre",
    "date_prelevement"
])
after_dedup = df_clean.count()
print(f"✓ Deduplication: {before_dedup} → {after_dedup} records ({before_dedup - after_dedup} duplicates removed)")

# 3. Handle nulls
before_nulls = df_clean.count()
df_clean = df_clean.filter(
    F.col("date_prelevement").isNotNull() &
    F.col("code_commune").isNotNull() &
    F.col("code_parametre").isNotNull()
)
after_nulls = df_clean.count()
print(f"✓ Null filtering: {before_nulls} → {after_nulls} records ({before_nulls - after_nulls} null records removed)")

# 4. Standardize column names
df_silver = df_clean \
    .withColumnRenamed("code_departement", "department_code") \
    .withColumnRenamed("nom_departement", "department_name") \
    .withColumnRenamed("code_prelevement", "sampling_code") \
    .withColumnRenamed("code_parametre", "parameter_code") \
    .withColumnRenamed("libelle_parametre", "parameter_name") \
    .withColumnRenamed("resultat_numerique", "numeric_result") \
    .withColumnRenamed("libelle_unite", "unit") \
    .withColumnRenamed("code_commune", "city_code") \
    .withColumnRenamed("nom_commune", "city_name") \
    .withColumnRenamed("date_prelevement", "sampling_date") \
    .withColumnRenamed("conclusion_conformite_prelevement", "compliance_conclusion")

print("✓ Columns standardized")

# 5. Add enrichments
df_silver = df_silver.withColumn(
    "quality_category",
    F.when(F.col("compliance_conclusion").contains("conforme aux exigences"), "compliant")
     .when(F.col("compliance_conclusion").contains("non conforme"), "non_compliant")
     .otherwise("unknown")
)

df_silver = df_silver \
    .withColumn("sampling_year", F.year("sampling_date")) \
    .withColumn("sampling_month", F.month("sampling_date"))

df_silver = df_silver.withColumn(
    "data_quality",
    F.when(F.col("numeric_result").isNotNull(), "valid")
     .otherwise("missing_value")
)

print("✓ Enrichment complete")

In [ ]:
# Show AFTER transformation
print("=== AFTER (Silver) ===")
print("\nSILVER Schema:")
df_silver.printSchema()

In [ ]:
# Sample transformed data
df_silver.select(
    "sampling_date",
    "sampling_year",
    "sampling_month",
    "city_name",
    "department_code",
    "department_name",
    "parameter_name",
    "numeric_result",
    "unit",
    "quality_category",
    "data_quality"
).show(10, truncate=False)

In [ ]:
# Statistics
print("=== Quality Categories Distribution ===")
df_silver.groupBy("quality_category").count().show()

print("=== Data Quality Distribution ===")
df_silver.groupBy("data_quality").count().show()

print("=== Records by Year ===")
df_silver.groupBy("sampling_year").count().orderBy("sampling_year").show()